In [1]:
from datasets import Dataset

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re


c:\Users\agsto\Desktop\CMPE-252-summary-project\summary_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import BartTokenizer, BartForConditionalGeneration, Seq2SeqTrainer,DataCollatorForSeq2Seq, Seq2SeqTrainingArguments
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())       # True if GPU is usable
print("CUDA version:", torch.version.cuda)                # CUDA version PyTorch was built with
print("Number of GPUs:", torch.cuda.device_count())       # Number of GPUs detected
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu126
CUDA available: True
CUDA version: 12.6
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 4070 SUPER


## Importing and cleaning data set

In [3]:
documention_path_0 = "document_train_0.parquet" #file path
document_0 = pd.read_parquet(documention_path_0)

documention_path_1 = "document_train_1.parquet"
document_1 = pd.read_parquet(documention_path_1)

documention_path_2 = "document_train_2.parquet"
document_2 = pd.read_parquet(documention_path_2)

documention_path_3 = "document_train_3.parquet"
document_3 = pd.read_parquet(documention_path_3)

In [ ]:
document = pd.concat([document_0, document_1, document_2, document_3])

In [ ]:
document

In [3]:
def clean_document(document):
    document['abstract'] = document['abstract'].str.replace(' \n', '', regex=False) #remove \n
    document['abstract'] = document['abstract'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
    document['abstract'] = document['abstract'].str.replace('  ', ' ', regex=False) #replace double space with single space

    document['article'] = document['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
    document['article'] = document['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

    article_summary = document['article'].str.len().describe()
    abstract_summary = document['abstract'].str.len().describe()

    document = document[document['article'].str.len() >= article_summary['25%']] # has to be at least 25% article length
    document = document[document['article'].str.len() <= article_summary['75%']] # has to be at most 75% article length

    document = document[document['abstract'].str.len() >= abstract_summary['25%']] 
    document = document[document['abstract'].str.len() <= abstract_summary['75%']]

    document = document[document['article'].str.len() >= article_summary['25%']] 
    document = document[document['article'].str.len() <= article_summary['75%']]  

    document = document[document['abstract'].str.len() >= abstract_summary['25%']]
    document = document[document['abstract'].str.len() <= abstract_summary['75%']]

    document = document.drop_duplicates()
    document = document.reset_index(drop=True)
    
    return document

In [ ]:
document = clean_document(document)

## Validation and Test cleaning

In [6]:
model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name, output_attentions=True, torch_dtype = torch.bfloat16)

The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 511/511 [00:00<00:00, 700.72it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


In [4]:
validation_path = "document_validation.parquet" #file path
validation = pd.read_parquet(validation_path)

test_path = "document_test.parquet"
test = pd.read_parquet(test_path)

In [6]:
validation = clean_document(validation)

In [5]:
test = clean_document(test)

## Training

In [ ]:
train_dataset = Dataset.from_pandas(document)
validation_dataset   = Dataset.from_pandas(validation)
test_dataset  = Dataset.from_pandas(test)

In [ ]:
label = tokenizer(
    list(train_dataset["abstract"]),
    truncation=True,
    max_length=1024,
    padding=False
)

# Get lengths
label_lengths = [len(ids) for ids in label["input_ids"]]

In [ ]:
abstract_tokens_summary = pd.DataFrame(label_lengths).describe()[0]
abstract_tokens_summary

In [ ]:
token_len_IQR =  abstract_tokens_summary['75%'] - abstract_tokens_summary['25%']
1.5 * token_len_IQR

In [ ]:
abstract_tokens_summary['25%'] - (1.5 * token_len_IQR)

In [ ]:
abstract_tokens_summary['75%'] + (1.5 * token_len_IQR)

#pad to 320 should be a clean number

In [ ]:
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-7746").cuda()
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-7746")

In [ ]:
def tokenize_document(dataset):
    model_input = tokenizer(dataset["article"], max_length = 1024, truncation = True)
    labels = tokenizer(dataset["abstract"], max_length = 320, truncation = True)

    model_input["labels"] = labels['input_ids']
    return model_input

In [ ]:
tokenized_dataset = train_dataset.map(tokenize_document, batched=True)
validation_tokenized = validation_dataset.map(tokenize_document, batched=True, remove_columns=validation_dataset.column_names)
test_tokenized = test_dataset.map(tokenize_document, batched=True, remove_columns=test_dataset.column_names)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest"  # pads to the longest sequence in each batch
)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_finetuned",      # folder to save checkpoints
    num_train_epochs=3,                  # number of passes through the dataset
    per_device_train_batch_size=4,       # safe for 12GB VRAM
    per_device_eval_batch_size=4,        # same as train batch size
    learning_rate=2e-5,                  # learning rate
    weight_decay=0.01,                   # regularization
    save_total_limit=3,                   # keep last 3 checkpoints
    eval_strategy="epoch",          # evaluate once per epoch
    save_strategy="epoch",                # save checkpoint once per epoch
    logging_strategy="steps",             # log every N steps
    logging_steps=100,                    # adjust if needed
    bf16=True,                            # mixed precision for VRAM efficiency
    predict_with_generate=True            # necessary for seq2seq evaluation
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=validation_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train(resume_from_checkpoint="./bart_finetuned/checkpoint-7746") #pick up from Epoch = 2

### Epoch	Training Loss	Validation Loss
### 1	2.795941	2.686332
### 2	2.850331	2.683255
### 3	2.777663	2.682559

# Summary Generation comparison

In [8]:
train_section = pd.read_parquet('section_train_4.parquet')

train_section['article'] = train_section['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
train_section['article'] = train_section['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

train_document = pd.read_parquet('document_train_4.parquet')

train_document['article'] = train_document['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
train_document['article'] = train_document['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

In [6]:
from transformers import BartForConditionalGeneration, BartTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-11619", attn_implementation="eager").to(device)
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-11619")

Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 512/512 [00:00<00:00, 1132.40it/s, Materializing param=model.shared.weight]                                   


In [9]:
test_ind = 4

In [10]:
inputs = tokenizer(
    train_document['article'][test_ind],
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=1024   # BART max input length
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        max_length=320,
        min_length=30,
        num_beams=1,
    )

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

The following generation flags are not valid and may be ignored: ['early_stopping', 'length_penalty']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


we investigate the case of cluster boundaries in schramm loewner evolution models with appropriate boundary conditions . the scaling limit of the spin cluster boundary in the ising model with appropriate boundaries is sle with . we simulate numerically a set of samples with these boundary conditions and compare to predictions from two - curve sle : the fractal dimensions and the formulae describing the expected spatial distribution of the curves in the bulk are compared against two predictions of single - curve schram m loewners . we also examine the statistics of the driving function by undoing the sequence of conformal transformations which determine the form of the curve through the curve . this driving function should be brownian motion with diffusivity curves , which correspond to sle . we find that the driving functions of the cluster boundaries are the same for all values of sle .


In [11]:
train_document['abstract'][test_ind]

'the scaling limit of the spin cluster boundaries of the ising model with domain wall boundary conditions is sle with @xmath0 . \n we hypothesise that the three - state potts model with appropriate boundary conditions has spin cluster boundaries which are also sle in the scaling limit , but with @xmath1 . to test this , we generate samples using the wolff algorithm and \n test them against predictions of sle : we examine the statistics of the loewner driving function , estimate the fractal dimension and test against schramm s formula . \n the results are in support of our hypothesis .'

In [12]:
train_document['article'][test_ind]

'much of the recent interest in schramm loewner evolution ( sle ) stems from the description it offers of cluster boundaries in critical statistical mechanics models . these models are conjecturally described by conformal field theories , which determines expectation values of products of local operators . however , non - local objects such as cluster boundaries do not have a natural expression in the field theory language . these curves are of great interest as , for example , they describe percolation cluster boundaries , level lines of height models and spin cluster boundaries . recently , the scaling limit of the spin cluster boundary in the ising model with appropriate boundary conditions has been proven to be sle with  . the ising model corresponds to the -state potts model with . we may ask , therefore , whether spin boundaries in the critical potts model with other values of have sle as their scaling limit . the fortuin - kastelyn ( fk ) cluster representation  of the potts mod

# Cross Attention

In [ ]:
def get_token_score_pairs(article, path = r"bart_finetuned\checkpoint-11619"):
    """"
    This function gives the importance score of each token. 
    Be mindful and use sectioned article with \n as we will use that to detect new lines

    Parameters
    ----------
    article: str
        the article we want to get the a summary from
    
    path: path
        path to the folder of pre trained model

    Returns
    -------
    token_score_pairs: list of tuples
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #prioritze cuda
    model = BartForConditionalGeneration.from_pretrained(path, attn_implementation="eager").to(device)
    tokenizer = BartTokenizer.from_pretrained(path)

    inputs = tokenizer(
        article,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=1024
    ).to(device)

    outputs = model(
        **inputs,
        output_attentions=True,
        return_dict=True
    )

    cross_attentions = outputs.cross_attentions #gets the cross attention scores for each token in the article
    last_layer = cross_attentions[-1] #looks at last layer
    avg_heads = last_layer.mean(dim=1) #gets the average of all heads of attention scores for each token 
    token_importance = avg_heads.mean(dim=1) #gives overall importance of each token

    input_ids = inputs["input_ids"][0]
    tokens = tokenizer.convert_ids_to_tokens(input_ids) #gives the tokens
    
    token_scores = [(token, float(score.detach())) for token, score in zip(tokens, token_importance[0])] #gives the list of tuples with tokens and importance score

    return token_scores
    

In [15]:
def rank_important_lines(token_scores, average_score = False):
    """"
    This function ranks lines based on their importance divided by the amount of tokens
    This only works if we use a sectioned article with \n as we will use that to detect new lines

    Parameters
    ----------
    token_scores: list of tuples (token, score)
        token_scores is a list of tuples where each tuple contains a token and its corresponding importance score
    
    average_score: bool (default False)
        if True then it will rank based on the average score of all tokens in a line
        if False then it will rank based on the sum of all tokens in a line

    Returns
    -------
    list of tuples (line, total_score, avg_score)
        this function returns a list of tuples where each
    """
    lines = []
    current_line = []
    current_score = 0

    for token, score in token_scores:
        if token != "Ċ": #this token symbolizes new line so if it isn't a new line keeping adding tokens
            current_line.append(token)
            current_score += score
        else:
            if current_line != []: #check if current line is empty or not
                avg_score = current_score / len(current_line) #calculate average score for the line
                lines.append((" ".join(current_line).replace("Ġ","").strip(' . '), current_score, avg_score)) 
            current_line = []
            current_score = 0

    if current_line != []:
        avg_score = current_score / len(current_line) #calculate average score for the line
        lines.append((" ".join(current_line).replace("Ġ","").strip(' . '), current_score, avg_score))

    lines[0]  = (lines[0][0].replace("<s>", "").strip(),  lines[0][1], lines[0][2]) #take out starting token
    lines[-1] = (lines[-1][0].replace("</s>", "").strip(), lines[-1][1], lines[-1][2]) #take out ending token

    if average_score:
        ranked_lines = sorted(lines, key=lambda x: x[2], reverse=True) #ranks the lines based on the average score in desc order
    else:
        ranked_lines = sorted(lines, key=lambda x: x[1], reverse=True) #ranks the lines based on the total score in desc order

    return ranked_lines

In [16]:
pairs = get_token_score_pairs(train_section['article'][test_ind])
rank_important_lines(pairs)

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 1163.31it/s, Materializing param=model.shared.weight]                                   


[('much of the recent interest in sch ram m lo ew ner evolution ( sle ) stems from the description it offers of cluster boundaries in critical statistical mechanics models',
  0.3201441764831543,
  0.009701338681307707),
 ('therefore , we have reason to believe that the f k cluster boundaries should correspond to sle in the scaling limit , with   the spin cluster boundaries are related to these f k cluster boundaries in a non - trivial way , however , and it is therefore interesting to investigate the properties of these spin boundaries in the scaling limit . for the is ing model , these spin cluster boundaries are also described by sle at the dual value',
  0.059539079666137695,
  0.0007087985674540201),
 ('if the boundary conditions are chosen as for the is ing model , with those on the left part being of spin type  ( say ) and those on the right of spin type , the right boundary curve of the cluster connected to the boundary spins of type  does not everywhere coincide with the right

In [17]:
print(summary)

we investigate the case of cluster boundaries in schramm loewner evolution models with appropriate boundary conditions . the scaling limit of the spin cluster boundary in the ising model with appropriate boundaries is sle with . we simulate numerically a set of samples with these boundary conditions and compare to predictions from two - curve sle : the fractal dimensions and the formulae describing the expected spatial distribution of the curves in the bulk are compared against two predictions of single - curve schram m loewners . we also examine the statistics of the driving function by undoing the sequence of conformal transformations which determine the form of the curve through the curve . this driving function should be brownian motion with diffusivity curves , which correspond to sle . we find that the driving functions of the cluster boundaries are the same for all values of sle .


In [18]:
train_document['abstract'][test_ind]

'the scaling limit of the spin cluster boundaries of the ising model with domain wall boundary conditions is sle with @xmath0 . \n we hypothesise that the three - state potts model with appropriate boundary conditions has spin cluster boundaries which are also sle in the scaling limit , but with @xmath1 . to test this , we generate samples using the wolff algorithm and \n test them against predictions of sle : we examine the statistics of the loewner driving function , estimate the fractal dimension and test against schramm s formula . \n the results are in support of our hypothesis .'

In [19]:
train_document['article'][test_ind]

'much of the recent interest in schramm loewner evolution ( sle ) stems from the description it offers of cluster boundaries in critical statistical mechanics models . these models are conjecturally described by conformal field theories , which determines expectation values of products of local operators . however , non - local objects such as cluster boundaries do not have a natural expression in the field theory language . these curves are of great interest as , for example , they describe percolation cluster boundaries , level lines of height models and spin cluster boundaries . recently , the scaling limit of the spin cluster boundary in the ising model with appropriate boundary conditions has been proven to be sle with  . the ising model corresponds to the -state potts model with . we may ask , therefore , whether spin boundaries in the critical potts model with other values of have sle as their scaling limit . the fortuin - kastelyn ( fk ) cluster representation  of the potts mod

## ROUGE comparison

1. Have the pre trained model and fine tuned model and both should use the same parameters
2. input cleaned or entire dataset into both params
3. calculate rouge 1, rouge 2, rouge L, and rouge L sum for each model
4. take average scores across all generations to get averaged Rouge 1, averaged Rouge ..., and compare

In [20]:
test['article'][0]

"recently it was discovered that feynman integrals obey functional equations , . different examples of functional equations were presented in refs . , , . in these articles only one - loop integrals were considered .  in the present paper we propose essentially new methods for deriving functional equations . these methods are based on algebraic relations between propagators and they are suitable for deriving functional equations for multi - loop integrals . also these methods can be used to derive functional equations for integrals with some propagators raised to non - integer powers . our paper is organized as follows . in sec . 2 . the method proposed in ref .  is shortly reviewed .  in sec . 3 . a method for finding algebraic relations between products of propagators is formulated . we describe in detail derivation of explicit relations for products of two , three and four propagators . also algebraic relation for products of arbitrary number of proparators is given . these relation

In [21]:
test['abstract'][0]

'new methods for obtaining functional equations for feynman integrals are presented . application of these methods for finding functional equations for various one- and two- loop integrals described in detail . it is shown that with the aid of functional equations feynman integrals in general kinematics can be expressed in terms of simpler integrals . pacs numbers : 02.30.gp , 02.30.ks , 12.20.ds , 12.38.bx + keywords : feynman integrals , functional equations   +  derivation of functional equations for feynman integrals + from algebraic relations  +  * o.v . tarasov * +  ii . institut fr theoretische physik , universitt hamburg , + luruper chaussee 149 , 22761 hamburg , germany + and + joint institute for nuclear research , + 141980 dubna , russian federation + : otarasov.ru +'

In [22]:
from evaluate import load

In [23]:
from transformers import BartForConditionalGeneration, BartTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-11619").to(device)
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-11619")

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 1152.66it/s, Materializing param=model.shared.weight]                                   


In [ ]:
import torch
from evaluate import load
from transformers import BartForConditionalGeneration, BartTokenizer

# Device and model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "facebook/bart-large-cnn"
model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
tokenizer = BartTokenizer.from_pretrained(model_name)

articles = test['article'].to_list()
abstracts = test['abstract'].to_list()

rouge = load("rouge")

batch_size = 2
summaries = []

for i in range(0, len(articles), batch_size):
    batch_articles = articles[i:i+batch_size]

    inputs = tokenizer(
        batch_articles,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=1024  # BART input limit
    ).to(device)

    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_length=320,
            min_length=30,
            num_beams=1
        )

    batch_summaries = [tokenizer.decode(s, skip_special_tokens=True) for s in summary_ids]

    summaries.extend(batch_summaries)

results = rouge.compute(predictions=summaries, references=abstracts)
print(results)

Loading weights: 100%|██████████| 511/511 [00:00<00:00, 1182.57it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


{'rouge1': np.float64(0.31840859170721303), 'rouge2': np.float64(0.09706548835740322), 'rougeL': np.float64(0.18876960454777048), 'rougeLsum': np.float64(0.18879169146238267)}


## For Epoch = 3 on testing data

{'rouge1': np.float64(0.4255260716239057),
 'rouge2': np.float64(0.15537410540264113),
 'rougeL': np.float64(0.24006861503538948),
 'rougeLsum': np.float64(0.24007864179224242)}

## For base Bart model

{'rouge1': np.float64(0.31840859170721303), 'rouge2': np.float64(0.09706548835740322), 'rougeL': np.float64(0.18876960454777048), 'rougeLsum': np.float64(0.18879169146238267)}


In [31]:
results

{'rouge1': np.float64(0.31840859170721303),
 'rouge2': np.float64(0.09706548835740322),
 'rougeL': np.float64(0.18876960454777048),
 'rougeLsum': np.float64(0.18879169146238267)}

In [133]:
from evaluate import load

# Load the 'rouge' metric
rouge = load('rouge')

# Define your predictions and references (can be lists for multiple examples)
predictions = [summary]
references = [test['abstract'][0]]

# Compute the scores
results = rouge.compute(predictions=predictions, references=references)

# Print the scores
print(results)

{'rouge1': np.float64(0.34123222748815163), 'rouge2': np.float64(0.08612440191387559), 'rougeL': np.float64(0.17061611374407581), 'rougeLsum': np.float64(0.17061611374407581)}


In [134]:
model_name = "facebook/bart-large-cnn"
model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
tokenizer = BartTokenizer.from_pretrained(model_name)

Loading weights: 100%|██████████| 511/511 [00:00<00:00, 1118.10it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


In [ ]:
inputs = tokenizer(
    test['article'][0],
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=1024   # BART max input length
).to(device)

with torch.no_grad():
    summary_ids = model.generate(
        **inputs,
        max_length=320,
        min_length=30,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

In the present paper we propose essentially new methods for deriving functional equations. These methods are based on algebraic relations between propagators. They can be used to derive functional equations for integrals with some propagators raised to non - integer powers. In conclusion we formulate our vision of the future applications and developments.


In [136]:
from evaluate import load

# Load the 'rouge' metric
rouge = load('rouge')

# Define your predictions and references (can be lists for multiple examples)
predictions = [summary]
references = [test['abstract'][0]]

# Compute the scores
results = rouge.compute(predictions=predictions, references=references)

# Print the scores
print(results)

{'rouge1': np.float64(0.2732919254658385), 'rouge2': np.float64(0.10062893081761007), 'rougeL': np.float64(0.18633540372670807), 'rougeLsum': np.float64(0.18633540372670807)}
